# 01. Data Understanding and Schema Audit
This notebook inspects all 9 raw Olist datasets, their grain, primary/foreign keys, missing values, and multi-row relationships.

In [1]:
import os
from pathlib import Path
if Path.cwd().name == 'notebooks':
    os.chdir(Path.cwd().parent)
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.data.load_data import load_raw_datasets

datasets = load_raw_datasets("data/raw")
summary_rows = []
for name, df in datasets.items():
    summary_rows.append({
        "dataset": name,
        "rows": len(df),
        "cols": len(df.columns),
        "exact_duplicates": int(df.duplicated().sum()),
        "total_nulls": int(df.isnull().sum().sum())
    })
print(pd.DataFrame(summary_rows).to_string(index=False))

             dataset    rows  cols  exact_duplicates  total_nulls
              orders   99441     8                 0         4908
         order_items  112650     7                 0            0
      order_payments  103886     5                 0            0
       order_reviews   99224     7                 0       145903
           customers   99441     5                 0            0
             sellers    3095     4                 0            0
            products   32951     9                 0         2448
category_translation      71     2                 0            0
         geolocation 1000163     5            261831            0


## Multi-Table Grain & Relationship Audit
Before joining tables to `orders`, we verify which tables contain multiple rows per `order_id` (`order_items`, `order_payments`, `order_reviews`) and check `geolocation` zip prefix duplicates.

In [2]:
orders = datasets["orders"]
items = datasets["order_items"]
payments = datasets["order_payments"]
reviews = datasets["order_reviews"]

print("Order status distribution:")
print(orders["order_status"].value_counts().to_string())
print("\nMulti-record grain check:")
print("Orders with >1 line item:", int((items.groupby("order_id").size() > 1).sum()))
print("Orders with >1 seller:", int((items.groupby("order_id")["seller_id"].nunique() > 1).sum()))
print("Orders with >1 payment row:", int((payments.groupby("order_id").size() > 1).sum()))
print("Orders with >1 review row:", int((reviews.groupby("order_id").size() > 1).sum()))

Order status distribution:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2

Multi-record grain check:
Orders with >1 line item: 9803
Orders with >1 seller: 1278


Orders with >1 payment row: 2961
Orders with >1 review row: 547
